## 06 - Lines and Distances

### Goal

- Last lesson we loaded the WDO package and used some of the given spatial functions. Now we are going to put package to use and calculate many distances and draw some lines. 
- Using any file that has "world cities" in the name:
  
  - [WorldCitiesGeo](../data/WorldCitiesGeo/) (and all its geojson files in the directory)
  - [world_cities_fixed.json](../data/world_cities_fixed.json)
  - [world_cities_by_time-zone.json](../data/world_cities_by_time-zone.json)
  
- These contain ALL THE CITIES IN THE WORLD! Well ... most of the cities.  You know like the ones that ... have people.  I mean `Burkburnett` probably isn't in the file, but it does have people, so ... (update ... I just checked, and Burk is in the file!)
  
- Keep reading


### Tasks

- Read in a file containing the locations to cities all over the world ([world_cities_large.json](./../data/world_cities_large.json)).
- We would like to draw a line from MSU (or close to it) to each city in the file, but thats not feasible as the file is too large. 
- Looking at the options (files) above, you should have plenty of choices for filtering down to a manageable size of cities to draw lines to. 


## Possible Bonus

- Randomly choose cities from the file, and stop choosing when the distance goes over some threshold (e.g 10000km).

In [12]:
from pathlib import Path
import sys
import math

GEO_OK = False

# First attempt: normal import
try:
    from wdo.geo import haversine_km, destination_point

    GEO_OK = True
except (ImportError, ModuleNotFoundError) as e:
    print(f"[WARN] Optional geo functions unavailable: {e}")

# Fallback: add src to sys.path, then retry
if not GEO_OK:
    print("[WARN] Trying src-path fallback (temporary workaround).")

    p = Path.cwd().resolve()
    repo_root = next(
        (c for c in [p, *p.parents] if (c / "pyproject.toml").exists()), None
    )

    if repo_root is not None:
        src_path = repo_root / "src"
        if src_path.exists() and str(src_path) not in sys.path:
            sys.path.insert(0, str(src_path))

        try:
            from wdo.geo import haversine_km, destination_point

            GEO_OK = True
            print("[INFO] Loaded wdo via src-path fallback.")
        except (ImportError, ModuleNotFoundError) as e:
            print(f"[WARN] Fallback import failed: {e}")

# Final no-crash fallback functions
if not GEO_OK:

    def haversine_km(*args, **kwargs):
        return None

    def destination_point(*args, **kwargs):
        return None



## Use last items learned from last lesson to locate and check if the file exists 
## Remember parent, and folder: data

# ----------------------------------------
# Distance: Haversine (km)
# ----------------------------------------
# def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
#     """
#     Compute great-circle distance between two lat/lon points.
#     Returns distance in kilometers.
#     Inputs: degrees
#     Output: kilometers
#     """
#     phi1 = math.radians(lat1)
#     phi2 = math.radians(lat2)

#     dphi = math.radians(lat2 - lat1)
#     dlmb = math.radians(lon2 - lon1)

#     a = sin(dphi / 2) ** 2 + cos(phi1) * cos(phi2) * sin(dlmb / 2) ** 2
#     c = 2 * atan2(sqrt(a), sqrt(1 - a))

#     return EARTH_RADIUS_KM * c



debug = False

cwd = Path.cwd()
if debug: 
    print("Current Working Directory:")
    print(cwd)

target = cwd.parent / "data" / "countries.geojson"
if debug: 
    print("\nAttempting to access:")
    print(target)

exists = target.exists()
if debug: 
    print("\nDoes file exist?")
    print(exists)

assert exists, (
    f"\n❌ ERROR: File not found:\n{target}\n"
    "Check spelling and folder structure."
)

print("Dallas → Houston (km):", haversine_km(32.7767, -96.7970, 29.7604, -95.3698))



Dallas → Houston (km): 361.7764642928539


**FYI:**

These previous three lessons will be helpful: 
- [03-Style_W_Logic](./03-Style_W_Logic.ipynb)
- [04-Distance](./04-Distance.ipynb)
- [05-Geo_and_Json_overview](./05-Geo_and_Json_overview.ipynb)



In [19]:
from pathlib import Path
import sys
import math
import json

####################################
# Copied code from Micro Lesson 04 #
####################################
GEO_OK = False

# First attempt: normal import
try:
    from wdo.geo import haversine_km, destination_point
    GEO_OK = True
except (ImportError, ModuleNotFoundError) as e:
    print(f"[WARN] Optional geo functions unavailable: {e}")

# Fallback: add src to sys.path, then retry
if not GEO_OK:
    print("[WARN] Trying src-path fallback (temporary workaround).")

    p = Path.cwd().resolve()
    repo_root = next(
        (c for c in [p, *p.parents] if (c / "pyproject.toml").exists()), None
    )

    if repo_root is not None:
        src_path = repo_root / "src"
        if src_path.exists() and str(src_path) not in sys.path:
            sys.path.insert(0, str(src_path))

        try:
            from wdo.geo import haversine_km, destination_point
            GEO_OK = True
            print("[INFO] Loaded wdo via src-path fallback.")
        except (ImportError, ModuleNotFoundError) as e:
            print(f"[WARN] Fallback import failed: {e}")

# Final no-crash fallback functions
if not GEO_OK:
    def haversine_km(lat1, lon1, lat2, lon2):
        # simple fallback implementation
        R = 6371
        phi1, phi2 = math.radians(lat1), math.radians(lat2)
        dphi = math.radians(lat2 - lat1)
        dlambda = math.radians(lon2 - lon1)

        a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
        return R * c

    def destination_point(*args, **kwargs):
        return None



# Locate data/world_cities_large.json safely
debug = False

cwd = Path.cwd()

# search upward for repo root, then into data/
repo_root = next((c for c in [cwd, *cwd.parents] if (c / "data").exists()), cwd)
city_file = repo_root / "data" / "world_cities_fixed.json"

if debug:
    print("Looking for:", city_file)

assert city_file.exists(), f"❌ Missing file: {city_file}"

print("[INFO] Loading city dataset...")

with open(city_file, "r", encoding="utf-8") as f:
    cities = json.load(f)

print(f"[INFO] Total cities loaded: {len(cities)}")



# Reference point: MSU Texas

msu_lat, msu_lon = 33.87, -98.51



# Limit by distance (recommended)
MAX_KM = 500  # keep manageable set

# ChatGPT Assisted with the logic here
filtered = []
for c in cities:
    lat = c.get("lat")
    lon = c.get("lng") or c.get("lon")

    if lat is None or lon is None:
        continue

    d = haversine_km(msu_lat, msu_lon, float(lat), float(lon))

    if d is not None and d <= MAX_KM:
        filtered.append(c)

print(f"[INFO] Filtered cities: {len(filtered)}")


# -------------------------------------------------------
# Build GeoJSON LineString features
# -------------------------------------------------------
geojson = {
    "type": "FeatureCollection",
    "features": []
}

for c in filtered:
    lat = float(c["lat"])
    lon = float(c.get("lng") or c.get("lon"))
    name = c.get("city", "Unknown")

    feature = {
        "type": "Feature",
        "properties": {
            "from-city": "MSU",
            "to-city": name,
            "stroke": "#ff8800"
        },
        "geometry": {
            "type": "LineString",
            "coordinates": [
                [msu_lon, msu_lat],
                [lon, lat]
            ]
        }
    }

    geojson["features"].append(feature)


print("[INFO] GeoJSON lines created:", len(geojson["features"]))


# Save output
output_file = repo_root / "data" / "msu_city_lines.geojson"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(geojson, f, indent=2)

print("[INFO] Saved:", output_file)


[INFO] Loading city dataset...
[INFO] Total cities loaded: 149102
[INFO] Filtered cities: 1092
[INFO] GeoJSON lines created: 1092
[INFO] Saved: /home/rjae/4545-Spatial-Data/Assignments/02-Missile_Geometry_101/data/msu_city_lines.geojson


This lesson helps with **📏 Milestone 2** 